## Load Data

In [ ]:
import pickle
import torch
import numpy as np

In [ ]:
def masked_mse(preds, labels, null_val):
    if torch.isnan(null_val):
        mask = ~torch.isnan(labels)
    else:
        mask = (labels != null_val)
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = (preds - labels)**2
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def masked_rmse(preds, labels, null_val):
    return torch.sqrt(masked_mse(preds=preds, labels=labels, null_val=null_val))


def masked_mae(preds, labels, null_val):
    if torch.isnan(null_val):
        mask = ~torch.isnan(labels)
    else:
        mask = (labels != null_val)
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = torch.abs(preds - labels)
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def masked_mape(preds, labels, null_val):
    if torch.isnan(null_val):
        mask = ~torch.isnan(labels)
    else:
        mask = (labels != null_val)
    mask = mask.float()
    mask /= torch.mean((mask))
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)
    loss = torch.abs(preds - labels) / labels
    loss = loss * mask
    loss = torch.where(torch.isnan(loss), torch.zeros_like(loss), loss)
    return torch.mean(loss)


def compute_all_metrics(preds, labels, null_val):
    mae = masked_mae(preds, labels, null_val).item()
    mape = masked_mape(preds, labels, null_val).item()
    rmse = masked_rmse(preds, labels, null_val).item()
    return mae, mape, rmse

In [ ]:
datasets_list = ['pems03_flow', 'pems04_flow', 'pems07_flow', 'pems08_flow', 'occpairs_occupancy', 'occhamburg_occupancy', 'pemsbay_speed', 'metrla_speed', 'trafficsh_speed', 'bikenyc_inflow', 'taxinyc_inflow', 'tdrive_inflow']
folder_path = "/data/weichen/ST-Library/datasets/eval_datasets"

In [ ]:
for few_shot_ratio in [0.1, 1.0]:
    print('-'*150)
    if few_shot_ratio == 0.1:
        print(f"\t\t\t\t\t\t eval_shot: Few-shot")
    else:
        print(f"\t\t\t\t\t\t eval_shot: Full-shot")
    print('-'*150)
    for dataset in datasets_list:
        
        print('*'*30)
        print(f"Dataset: {dataset}")
        print('*'*30)
        
        for num_steps in [12, 24]:
        
            with open(f"{folder_path}/{dataset}/{dataset}_temporal.pkl", 'rb') as f:
                df = pickle.load(f)
                raw_temporal_data = torch.tensor(df.values).unsqueeze(-1)
                f.close()

            T, N, _ = raw_temporal_data.shape

            # Add time-based features if specified
            feature_list = [raw_temporal_data]

            # add_time_of_day
            time_ind = (df.index.values - df.index.values.astype('datetime64[D]')) / np.timedelta64(1, 'D') * 288
            time_of_day = np.tile(time_ind, [1, N, 1]).transpose((2, 1, 0))
            feature_list.append(torch.tensor(time_of_day, dtype=torch.float32))

            # add_day_of_week:
            dow = df.index.dayofweek
            dow_tiled = np.tile(dow, [1, N, 1]).transpose((2, 1, 0))
            day_of_week = dow_tiled / 7 * 7
            feature_list.append(torch.tensor(day_of_week, dtype=torch.float32))

            temporal_data = torch.cat(feature_list, dim=-1).numpy()  # Concatenate features along the channel dimension

            # train_rate = few_shot_ratio if few_shot_ratio <= train_val_test_rate[0] else train_val_test_rate[0]
            train_rate = 0.6 * few_shot_ratio
            valid_rate = 0.2
            test_rate = 0.2

            train = temporal_data[:int(T * train_rate)]
            test = temporal_data[int(T * (1 - test_rate)):]
            
            # Create a dictionary to store lists of observations for each (tod, dow) pair
            history = {}
            # Iterate through the historical data to populate the history dictionary
            for i in range(train.shape[0]-num_steps-1):
                
                key = (train[i, 0, 1], train[i, 0, 2])  # (tod, dow) tuple
                if key not in history:
                    history[key] = []
                    
                # Collect observations for the next num_steps steps
                future_values = train[i+1:i+num_steps+1,:,0]
                
                history[key].append(future_values)  # Collect and store future values
            
            history_averages = {}

            # Calculate the average values for each (tod, dow) pair
            for k, v in history.items():
                stacked_array = np.stack(v, axis=-1)
                averages = np.mean(stacked_array, axis=-1)
                history_averages[k] = averages
            
            
            preds = []
            labels = []

            # Retrieve labels and predictions
            for i in range(test.shape[0]-num_steps-1):

                key = (test[i, 0, 1], test[i, 0, 2])  # (tod, dow) tuple
                
                labels.append(test[i+1:i+num_steps+1,:,0])
                
                try:
                    preds.append(history_averages[key])
                except KeyError:
                    # If the (tod, dow) pair is not present in the history dictionary, predict last point
                    preds.append(torch.tensor(test[i,:,0]).expand(num_steps, -1))
                
            labels = np.stack(labels, axis=0)
            preds = np.stack(preds, axis=0)
            
            labels = torch.Tensor(labels)
            preds = torch.Tensor(preds)

            # handle the precision issue when performing inverse transform to label
            mask_value = torch.tensor(0)

            test_mae = []
            test_mape = []
            test_rmse = []

            # Calculate metrics
            for i in range(12):
                res = compute_all_metrics(preds[:,i,:], labels[:,i,:], mask_value)
                test_mae.append(res[0])
                test_mape.append(res[1] * 100)
                test_rmse.append(res[2])

            mae_mean = np.mean(test_mae)
            mae_std = 0
            rmse_mean = np.mean(test_rmse)
            rmse_std = 0
            mape_mean = np.mean(test_mape)
            mape_std = 0
            
            if num_steps == 12:
                print('HA - Short Forecasting' + "\t\t MAE:" + "& $" + f"{mae_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mae_std:.2f}'+"}}$" + "\t\t RMSE:" + "& $" + f"{rmse_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{rmse_std:.2f}'+"}}$" + "\t\t MAPE:" + "& $" + f"{mape_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mape_std:.2f}'+"}}$")
            else:
                print('HA - Long Forecasting' + "\t\t MAE:" + "& $" + f"{mae_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mae_std:.2f}'+"}}$" + "\t\t RMSE:" + "& $" + f"{rmse_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{rmse_std:.2f}'+"}}$" + "\t\t MAPE:" + "& $" + f"{mape_mean:.2f}" + "\\textcolor{gray}{\\text{\scriptsize±" + f'{mape_std:.2f}'+"}}$")